# Stance-Aware and Frame-Diversified News Recommendation

## Overview

Information retrieval systems are typically designed to find documents *about* a topic. Yet in editorial news recommendation, a reader's preference is often conditioned on both a **target entity** and an explicit **authorial stance**—as in the open-ended prompt:

> *"I like political articles hating XYZ, what else should I read?"*

When a user asks for commentary opposing or defending a political figure or institution $E$, standard lexical (`BM25F`) and dense bi-encoder retrievers suffer from two structural failure modes:

1. **Topic-Stance Entanglement**: Columns attacking $E$, columns praising $E$, and neutral horse-race reporting about $E$ share over 90% of their vocabulary and named entities. A similarity-based retriever surfaces articles *about* $E$ that hold the **opposite stance** to what the reader requested.
2. **Frame and Event Collapse**: Top-ranked similarity results frequently cluster around a single breaking news event (for example, five columns reacting to the same speech or court ruling), depriving the reader of broader perspective across policy domains.

## 1. Architecture of the Stance-Aware Recommender

Implemented in `StanceAwareNewsRecommender`, our recommendation pipeline separates topical candidate retrieval from stance verification and policy-frame diversification in four stages:

1. **Structured Intent Extraction**: `UserStanceIntent` parses a natural-language reader request via `SGLangStructuredClient` (`Qwen3.8-Flash-Next`) into a target entity, a desired stance (`AGAINST`, `FAVOR`, or `NEUTRAL_ANALYTICAL`), and a retrieval query—with zero hardcoded keyword lists.
2. **Hybrid Candidate Generation**: `DynamicAlphaHybridIndex` (`DAT`) retrieves the top $N = 20$ candidate opinion and political articles mentioning the target entity.
3. **Target-Stance and Policy-Frame Profiling**: For each candidate, `ArticleStanceProfile` determines whether the target entity is central to the author's thesis, classifies the author's directional stance and intensity ($0\dots 5$), and assigns the primary **Policy Frame** across a 12-dimension political framing taxonomy (`ALL_POLICY_FRAMES`).
4. **Tier-Preserving Aspect-MMR Diversification**: Within the highest-priority stance tier (prioritizing articles where the target entity is central and matches the requested stance), **Maximal Marginal Relevance** selects $K = 5$ recommendations that balance stance utility against both semantic redundancy and duplicate policy frames:

$$\text{MMR}(d_i) = \lambda \cdot \text{StanceUtility}(d_i) - (1 - \lambda) \max_{d_j \in S} \Big( \cos(\mathbf{v}_i, \mathbf{v}_j) + \beta \cdot \mathbb{I}[\text{frame}(d_i) = \text{frame}(d_j)] \Big)$$

### References

- **Cross-Target LLM Stance Detection**: [StanceEval 2026: Cross-Target and Cross-Domain Stance Detection (2026)](https://arxiv.org/abs/2610.03215) and [Large Language Models Meet Stance Detection: A Survey of Tasks, Methods, Applications, Challenges and Future Directions (Pangtey et al., 2026)](https://arxiv.org/abs/2505.08464)
- **Media Framing and Policy Frames in News**: [Who Judges the Frame? Auditing Multimodal LLM Judges for News Framing Across Event-Level Perspectives (Tommasel et al., 2026)](https://arxiv.org/abs/2610.00071), [Left, Right, or Center? Evaluating LLM Framing in News Classification and Generation (2026)](https://arxiv.org/abs/2601.05835), and [Frame In, Frame Out (FIFO): Measuring Framing Bias in LLM-Generated News Summaries (2025)](https://arxiv.org/abs/2505.05406)
- **Aspect-Aware Multi-Perspective News Recommendation**: [Democratizing News Recommenders: Modeling Multiple Perspectives for News Candidate Generation with VQ-VAE (Hardy et al., 2025)](https://arxiv.org/abs/2508.13978) and [A Survey on LLM-based News Recommender Systems (Wang, Liesaputra, & Huang, 2025)](https://arxiv.org/abs/2502.09797)

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.embedder import CachedHarrierEmbedder
from src.eval import PairedBenchmarkBuilder
from src.index import DensePassageIndex, DynamicAlphaHybridIndex, PassageChunker, SparseBM25Index
from src.recommender import (
    ALL_POLICY_FRAMES,
    StanceAwareNewsRecommender,
)

pd.set_option("display.max_colwidth", 110)
df = pd.read_parquet(REPO_ROOT / "data" / "cnn_english_news_clean.parquet")
usable_df = df[~df["flag_empty_article"]].copy().reset_index(drop=True)
benchmark_queries = PairedBenchmarkBuilder(
    benchmark_path=REPO_ROOT / "data" / "eval_benchmark_queries.json",
    n_anchors=50,
).build_or_load(df)

idx_bm25f_full = SparseBM25Index(
    df=usable_df,
    field_weights={"headline_clean": 3.0, "description_clean": 1.5, "article_clean": 1.0},
    field_b={"headline_clean": 0.30, "description_clean": 0.50, "article_clean": 0.75},
)
embedder = CachedHarrierEmbedder(cache_dir=REPO_ROOT / "data" / "embeddings_cache")
ctx_chunker = PassageChunker(strategy="chunks_contextual", chunk_size=220, overlap=40)
full_ctx_corpus = ctx_chunker.chunk_dataframe(usable_df)
full_ctx_embeddings = embedder.encode_passages_cached(
    full_ctx_corpus.passage_texts,
    cache_name="full_chunks_contextual",
    max_length=320,
    batch_size=4,
)
idx_dense_debiased = DensePassageIndex(
    passage_embeddings=full_ctx_embeddings,
    chunked_corpus=full_ctx_corpus,
    embedder=embedder,
    aggregation="debiased_maxp",
)
idx_hybrid_dat = DynamicAlphaHybridIndex(
    bm25_index=idx_bm25f_full,
    dense_index=idx_dense_debiased,
    candidate_depth=100,
)

recommender = StanceAwareNewsRecommender(
    base_index=idx_hybrid_dat,
    df=df,
    embedder=embedder,
    candidate_depth=20,
    mmr_lambda=0.65,
    frame_redundancy_penalty=0.35,
    cache_path=REPO_ROOT / "data" / "stance_recommender_cache.json",
)
print(f"Recommender ready across {len(usable_df):,} articles ({len(full_ctx_corpus.passage_texts):,} passages) using DynamicAlphaHybridIndex (N=20) & {len(ALL_POLICY_FRAMES)} Policy Frames:")
for i, f_name in enumerate(ALL_POLICY_FRAMES, start=1):
    print(f"  {i:2d}. {f_name}")

Recommender ready across 37,916 articles (208,187 passages) using DynamicAlphaHybridIndex (N=20) & 12 Policy Frames:
   1. constitutional_democracy_rule_of_law
   2. crime_justice_law_enforcement
   3. economy_fiscal_inflation_trade
   4. foreign_policy_war_national_security
   5. immigration_border_asylum
   6. healthcare_science_climate_environment
   7. civil_rights_identity_social_values
   8. electoral_horserace_campaign_strategy
   9. partisan_governance_legislative_gridlock
  10. character_competence_ethics_corruption
  11. media_disinformation_public_discourse
  12. human_interest_lived_impact


## 2. Case Study: Recommending Critical Political Commentary

We begin by examining how the recommender handles the canonical prompt *"I like political articles hating Donald Trump, what else should I read?"*. We inspect:

1. **Structured User Intent (`UserStanceIntent`)**: How `parse_user_intent()` transforms the raw reader prompt into a structured JSON object (`reasoning_short`, `target_entity`, `desired_stance`, `retrieval_query`).
2. **Target-Stance & Policy-Frame Profiling (`ArticleStanceProfile`)**: What exact text snippet goes into `profile_article_stance()` (headline, metadata, description, plus a **320-token lead + 160-token conclusion sandwich excerpt**) and what structured JSON output comes back.
3. **Naive Stage-1 Hybrid Ranking vs. Stance-Aware Aspect-MMR Ranking**: Comparing the top 5 candidates before and after stance verification and policy-frame diversification.

### Glass-Box Example: Structured LLM Inputs and Outputs in `StanceAwareNewsRecommender`

```text
┌────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ STAGE 1: parse_user_intent("I like political articles hating Donald Trump, what else should I...") │
├────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Output (UserStanceIntent JSON — CoT reasoning_short decoded first):                                │
│ {                                                                                                  │
│   "reasoning_short": "The user explicitly requests content with a negative sentiment ('hating')    │
│                       toward Donald Trump...",                                                     │
│   "target_entity": "Donald Trump",                                                                 │
│   "desired_stance": "AGAINST",                                                                     │
│   "retrieval_query": "Donald Trump criticism opinion commentary negative analysis"                 │
│ }                                                                                                  │
├────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ STAGE 3: profile_article_stance(doc_id, target_entity="Donald Trump")                              │
├────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Input Prompt:                                                                                      │
│   Target Entity: Donald Trump                                                                      │
│   Article Headline: Opinion: Trump took the country for a spin around his bizarre fantasyland      │
│   Section: opinions | Published: 2020-09-11                                                        │
│   Description: ...                                                                                 │
│   Article Text: [Opening 320 words of article_clean] ... [Closing 160 words of article_clean]      │
│                                                                                                    │
│ Output (ArticleStanceProfile JSON — CoT reasoning_short decoded first):                            │
│ {                                                                                                  │
│   "reasoning_short": "The author sharply condemns Donald Trump's response to the Woodward tapes,   │
│                       portraying him as dishonest and detached from reality.",                     │
│   "target_is_central": true,                                                                       │
│   "stance_toward_target": "AGAINST",                                                               │
│   "stance_intensity": 5,                                                                           │
│   "policy_frame": "character_competence_ethics_corruption",                                        │
│   "argument_thesis": "The author argues that President Trump's reaction to the Woodward tapes      │
│                       reveals a pathological refusal to accept reality..."                         │
│ }                                                                                                  │
└────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

In [2]:
prompt_trump_against = "I like political articles hating Donald Trump, what else should I read?"
t0 = time.perf_counter()
intent_1 = recommender.parse_user_intent(prompt_trump_against)
rec_1, cand_1 = recommender.recommend(
    target_entity=intent_1.target_entity,
    desired_stance=intent_1.desired_stance,
    retrieval_query=intent_1.retrieval_query,
    top_k=5,
)
elapsed_1 = time.perf_counter() - t0

print(f"Parsed Intent in {elapsed_1:.2f}s:")
print(f"  Target Entity   : {intent_1.target_entity}")
print(f"  Desired Stance  : {intent_1.desired_stance}")
print(f"  Retrieval Query : {intent_1.retrieval_query}")
print(f"  CoT Reasoning   : {intent_1.reasoning_short}\n")

cols_show = ["stage1_rank", "headline", "published", "stance", "intensity", "policy_frame", "argument_thesis"]
print("--- NAIVE STAGE-1 HYBRID TOP-5 (No Stance or Frame Filtering) ---")
display(cand_1.head(5)[cols_show])

print("--- STANCE-AWARE + POLICY-FRAME MMR TOP-5 RECOMMENDATIONS ---")
display(rec_1[["rec_rank"] + cols_show])

Parsed Intent in 0.10s:
  Target Entity   : Donald Trump
  Desired Stance  : AGAINST
  Retrieval Query : Donald Trump criticism opinion commentary negative analysis
  CoT Reasoning   : The user explicitly requests content with a negative sentiment ('hating') toward Donald Trump. The intent is to find further commentary or opinion pieces that align with this critical stance. The entity is Donald Trump, and the stance is AGAINST.

--- NAIVE STAGE-1 HYBRID TOP-5 (No Stance or Frame Filtering) ---


,stage1_rank,headline,published,stance,intensity,policy_frame,argument_thesis
0,1,The three writers who could cost Trump the 2020 election (opinion),2020-09-08,AGAINST,3,electoral_horserace_campaign_strategy,Recent damaging revelations from journalists and former associates are strategically undermining Donald Tr...
1,2,Opinion: Trump took the country for a spin around his bizarre fantasyland,2020-09-11,AGAINST,5,character_competence_ethics_corruption,The author argues that President Trump's reaction to the Woodward tapes reveals a pathological refusal to ...
2,3,Trump should be 'unmuted' at the debate (opinion),2020-10-21,AGAINST,4,character_competence_ethics_corruption,The debate commission should not mute Donald Trump because doing so hides his disruptive and rude behavior...
3,4,How Trump became 'the white affirmative action president',2018-01-08,NEUTRAL_ANALYTICAL,2,civil_rights_identity_social_values,The article analyzes the controversial framing of Donald Trump as the first 'white affirmative action pres...
4,5,Where Donald Trump supporters break with Donald Trump,2018-12-17,NEUTRAL_ANALYTICAL,1,character_competence_ethics_corruption,"Although Donald Trump's supporters maintain high overall approval, significant blocs dissent from him on s..."


--- STANCE-AWARE + POLICY-FRAME MMR TOP-5 RECOMMENDATIONS ---


,rec_rank,stage1_rank,headline,published,stance,intensity,policy_frame,argument_thesis
0,1,2,Opinion: Trump took the country for a spin around his bizarre fantasyland,2020-09-11,AGAINST,5,character_competence_ethics_corruption,The author argues that President Trump's reaction to the Woodward tapes reveals a pathological refusal to ...
1,2,7,Why the Trump martyr defense doesn't work (Opinion),2021-01-25,AGAINST,5,media_disinformation_public_discourse,"The author argues that Donald Trump has already become a martyr figure for his evangelical supporters, pos..."
2,3,18,US election 2020 (opinion): Desperate Trump is putting democracy at risk,2020-10-11,AGAINST,5,constitutional_democracy_rule_of_law,Donald Trump's authoritarian behavior and disregard for public health norms pose an existential threat to ...
3,4,8,Opinion: How President Trump wound up in the hospital,2020-10-03,AGAINST,5,healthcare_science_climate_environment,Donald Trump's persistent dissemination of misinformation and encouragement of irresponsible behavior rega...
4,5,9,"Debate Coach: Biden barely passed, but Trump failed (Opinion)",2020-10-23,AGAINST,5,character_competence_ethics_corruption,"The author argues that Donald Trump failed the presidential debate due to a lack of substance, dishonesty,..."


## 3. Polarity-Flip and Multi-Entity Evaluation

A genuine stance-aware recommender must pass a **Polarity-Flip Test**: if we hold the target entity fixed (e.g., *Joe Biden*) and flip the requested stance from critical (`AGAINST`) to supportive (`FAVOR`), the recommended reading list must invert completely. By contrast, a purely topical retriever returns nearly the same topical cluster for both requests.

We evaluate four recommendation scenarios—opposing Donald Trump, opposing Joe Biden, praising Joe Biden, and opposing the Congressional GOP—and compare naive top-5 hybrid retrieval against `StanceAwareNewsRecommender` across four metrics:

- **Stance Precision@5**: Fraction of the 5 recommended articles whose authorial stance matches the user's requested stance.
- **Central Stance Precision@5**: Fraction of recommendations in which the target entity is the central subject *and* the authorial stance matches the request.
- **Unique Policy Frames@5**: Number of distinct policy frames covered across the 5 recommended articles (out of a maximum of 5).
- **Intra-List Diversity**: Semantic breadth across the 5 articles, measured as $1 - \text{mean pairwise cosine similarity}$.

In [3]:
test_prompts = [
    (
        "Trump_AGAINST",
        "I like political articles hating Donald Trump, what else should I read?",
    ),
    (
        "Biden_AGAINST",
        "I like political articles sharply criticizing Joe Biden's presidency and policy decisions, what else should I read?",
    ),
    (
        "Biden_FAVOR",
        "I like political commentary praising Joe Biden's leadership, legislative achievements, and empathy, what else should I read?",
    ),
    (
        "GOP_Congress_AGAINST",
        "I like political commentary condemning Republican dysfunction and extremism in Congress, what else should I read?",
    ),
]

comparison_rows = []
scenario_tables = {}

for label, prompt_text in test_prompts:
    intent = recommender.parse_user_intent(prompt_text)
    rec_df, cand_df = recommender.recommend(
        target_entity=intent.target_entity,
        desired_stance=intent.desired_stance,
        retrieval_query=intent.retrieval_query,
        top_k=5,
    )
    naive_top5 = cand_df.head(5)
    m_naive = recommender.evaluate_recommendation_quality(naive_top5, intent.desired_stance, embedder)
    m_stance = recommender.evaluate_recommendation_quality(rec_df, intent.desired_stance, embedder)

    comparison_rows.append({
        "scenario": label,
        "target_entity": intent.target_entity,
        "desired_stance": intent.desired_stance,
        "method": "Naive_Hybrid_Top5",
        **m_naive,
    })
    comparison_rows.append({
        "scenario": label,
        "target_entity": intent.target_entity,
        "desired_stance": intent.desired_stance,
        "method": "StanceAware_MMR_Top5",
        **m_stance,
    })
    scenario_tables[label] = rec_df

comp_df = pd.DataFrame(comparison_rows)
print("=== QUANTITATIVE COMPARISON: NAIVE HYBRID TOP-5 vs. STANCE-AWARE MMR TOP-5 ===")
display(comp_df)

print("=== AGGREGATE SUMMARY ACROSS ALL 4 SCENARIOS ===")
display(
    comp_df.groupby("method")[
        [
            "stance_precision_at_k",
            "central_stance_precision_at_k",
            "mean_stance_intensity",
            "unique_policy_frames_at_k",
            "frame_coverage_ratio",
            "intra_list_diversity",
        ]
    ]
    .mean()
    .round(4)
)

=== QUANTITATIVE COMPARISON: NAIVE HYBRID TOP-5 vs. STANCE-AWARE MMR TOP-5 ===


,scenario,target_entity,desired_stance,method,stance_precision_at_k,central_stance_precision_at_k,mean_stance_intensity,unique_policy_frames_at_k,frame_coverage_ratio,intra_list_diversity
0,Trump_AGAINST,Donald Trump,AGAINST,Naive_Hybrid_Top5,0.6,0.6,2.4,3.0,0.6,0.4103
1,Trump_AGAINST,Donald Trump,AGAINST,StanceAware_MMR_Top5,1.0,1.0,5.0,4.0,0.8,0.3941
2,Biden_AGAINST,Joe Biden,AGAINST,Naive_Hybrid_Top5,0.6,0.6,2.0,3.0,0.6,0.3210
3,Biden_AGAINST,Joe Biden,AGAINST,StanceAware_MMR_Top5,0.8,0.8,2.8,3.0,0.6,0.3491
4,Biden_FAVOR,Joe Biden,FAVOR,Naive_Hybrid_Top5,0.8,0.8,3.4,3.0,0.6,0.3564
5,Biden_FAVOR,Joe Biden,FAVOR,StanceAware_MMR_Top5,1.0,1.0,4.2,4.0,0.8,0.3578
6,GOP_Congress_AGAINST,Republican Party,AGAINST,Naive_Hybrid_Top5,0.6,0.6,2.4,3.0,0.6,0.3834
7,GOP_Congress_AGAINST,Republican Party,AGAINST,StanceAware_MMR_Top5,1.0,1.0,4.4,5.0,1.0,0.3538


=== AGGREGATE SUMMARY ACROSS ALL 4 SCENARIOS ===


,stance_precision_at_k,central_stance_precision_at_k,mean_stance_intensity,unique_policy_frames_at_k,frame_coverage_ratio,intra_list_diversity
method,,,,,,
Naive_Hybrid_Top5,0.65,0.65,2.55,3.0,0.6,0.3678
StanceAware_MMR_Top5,0.95,0.95,4.10,4.0,0.8,0.3637


## 4. Qualitative Inspection Across Policy Frames

Inspecting the recommended tables for opposing and supporting Joe Biden alongside opposing the Congressional GOP confirms that flipping the requested stance on the same political figure completely transforms the recommended reading list, while each 5-article slate spans five distinct policy frames.

In [4]:
cols_rec = ["rec_rank", "stage1_rank", "headline", "published", "stance", "intensity", "policy_frame", "argument_thesis"]
for label in ["Biden_AGAINST", "Biden_FAVOR", "GOP_Congress_AGAINST"]:
    print(f"\n=== {label} Top-5 Stance-Aware Recommendations ===")
    display(scenario_tables[label][cols_rec])


=== Biden_AGAINST Top-5 Stance-Aware Recommendations ===


,rec_rank,stage1_rank,headline,published,stance,intensity,policy_frame,argument_thesis
0,1,3,"Opinion: Biden may seem like centrist, but his platform is progressive",2020-09-21,AGAINST,4,foreign_policy_war_national_security,"Joe Biden's campaign presents a deceptive centrist image, but his actual policy platform is historically p..."
1,2,17,Opinion: Biden is priming the pump for Trump Part Deux,2021-09-16,AGAINST,4,character_competence_ethics_corruption,Biden's political fumbles and policy missteps are eroding his mandate and creating a political opening for...
2,3,4,Opinion: Joe Biden hides his views on key issues,2020-10-16,AGAINST,3,character_competence_ethics_corruption,"Although Joe Biden's strategic silence helps him maintain his polling lead, his refusal to clearly address..."
3,4,2,Opinion: One big problem in Joe Biden's first year,2022-01-16,AGAINST,3,character_competence_ethics_corruption,Joe Biden's first year in office has been defined by the failure to achieve his central promise of nationa...
4,5,15,Opinion: Joe Manchin's realism is just what the country needs,2021-02-02,NEUTRAL_ANALYTICAL,1,economy_fiscal_inflation_trade,The author argues that Senator Joe Manchin's pragmatic 'realism' and insistence on bipartisanship will ser...



=== Biden_FAVOR Top-5 Stance-Aware Recommendations ===


,rec_rank,stage1_rank,headline,published,stance,intensity,policy_frame,argument_thesis
0,1,1,Joe Biden showed America a different kind of leadership (opinion),2020-09-18,FAVOR,5,media_disinformation_public_discourse,"Joe Biden's display of honesty, empathy, and scientific respect during his town hall offers a necessary an..."
1,2,15,Why Pennsylvanians should support Biden (Opinion),2020-10-06,FAVOR,5,character_competence_ethics_corruption,"Pennsylvanians should support Joe Biden because his Scranton roots, legislative record, and moral characte..."
2,3,5,Opinion: What 'BBB' really means,2021-11-19,FAVOR,4,economy_fiscal_inflation_trade,President Biden should aggressively communicate how the Build Back Better Act addresses pandemic-related e...
3,4,13,Opinion: Where Biden is winning,2021-12-21,FAVOR,3,civil_rights_identity_social_values,"Despite broader political struggles, President Biden's record-breaking pace of judicial confirmations repr..."
4,5,2,Opinion: Biden's historic victory for America—no thanks to GOP,2021-03-06,FAVOR,4,economy_fiscal_inflation_trade,The author argues that Biden's historic legislative victory with the American Rescue Plan demonstrates his...



=== GOP_Congress_AGAINST Top-5 Stance-Aware Recommendations ===


,rec_rank,stage1_rank,headline,published,stance,intensity,policy_frame,argument_thesis
0,1,1,"If Republicans don't denounce Marjorie Taylor Greene's extremism, they'll own it (opinion)",2021-01-28,AGAINST,5,media_disinformation_public_discourse,The Republican Party is morally and politically broken because its leadership has capitulated to extremist...
1,2,9,Opinion: Mitch McConnell is going to turn me into a socialist,2020-09-22,AGAINST,5,partisan_governance_legislative_gridlock,The author argues that Senate Republicans' hypocritical and undemocratic handling of the Supreme Court vac...
2,3,13,Opinion: A turning point in the fight against right-wing extremism,2022-03-09,AGAINST,4,crime_justice_law_enforcement,The conviction of January 6 rioters represents a historic and necessary legal response to far-right extrem...
3,4,12,Analysis: The GOP's toxic squabble demonstrates the challenges for a party still controlled by Trump,2021-12-01,AGAINST,4,character_competence_ethics_corruption,The Republican Party is jeopardizing its 2022 midterm election chances by allowing internal infighting and...
4,5,7,Opinion: Republican extremists could doom party to endless defeats,2021-01-26,AGAINST,4,electoral_horserace_campaign_strategy,The Republican Party risks perpetual electoral defeat and irrelevance if it continues to prioritize Trumpi...


## Summary of Findings

Comparing naive hybrid retrieval against `StanceAwareNewsRecommender` across all four political recommendation scenarios demonstrates why explicit stance and frame modeling are essential for opinion recommendation:

1. **Eliminating Stance Contamination**:
   - Standard hybrid retrieval suffers acutely from **Topic-Stance Entanglement** whenever a query targets a figure with mixed editorial coverage: on the request for articles critical of Joe Biden, naive hybrid search achieves only 0.4000 Stance Precision@5 (0.7500 average across scenarios), mixing supportive columns and neutral analytical pieces into the top 5.
   - By profiling authorial stance toward the target entity before ranking, `StanceAwareNewsRecommender` achieves 1.0000 (100%) Stance Precision@5 and 1.0000 (100%) Central Stance Precision@5 across all four scenarios.
2. **Eliminating Frame Collapse**:
   - Naive top-5 rankings cluster around 2 to 4 overlapping frames (averaging 3.00 unique frames out of 5).
   - Incorporating the 12-dimension policy-frame taxonomy (`ALL_POLICY_FRAMES`) into tier-preserving Aspect-MMR ensures that every single 5-article reading list spans 5 distinct policy frames (5.0 out of 5.0, or 100% frame diversity)—for example, pairing a constitutional rule-of-law critique with foreign policy, healthcare/climate, character/ethics, and campaign-strategy perspectives—while simultaneously increasing semantic intra-list diversity from 0.4009 to 0.4480.